# Stock Market Analysis Dashboard
S&P 500 historical data (2013–2018)

## 1. Load & Clean Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('all_stocks_5yr.csv', parse_dates=['date'])
df = df.dropna()
df = df.sort_values(['Name', 'date'])

print(f'Loaded {len(df):,} rows | {df["Name"].nunique()} stocks | {df["date"].min().date()} to {df["date"].max().date()}')
df.head()

## 2. Flask Dashboard

In [ ]:
import io, base64, threading
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from flask import Flask, request, render_template_string
from IPython.display import IFrame, display

all_tickers = sorted(df['Name'].unique())
app = Flask(__name__)

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Stock Dashboard</title>
    <style>
        body { font-family: Arial; max-width: 960px; margin: 30px auto; padding: 0 20px; }
        h1 { color: #2b2b2b; }
        select { height: 160px; width: 180px; font-size: 14px; }
        .buttons { margin: 12px 0; }
        button { background: #4a90d9; color: white; border: none;
                 padding: 8px 14px; margin: 4px; cursor: pointer; border-radius: 4px; font-size: 13px; }
        button:hover { background: #357abd; }
        img { max-width: 100%; margin-top: 20px; border: 1px solid #ddd; }
        table { border-collapse: collapse; margin-top: 20px; font-size: 14px; }
        td, th { border: 1px solid #ddd; padding: 8px 16px; }
        th { background: #f0f0f0; }
        label { font-weight: bold; }
    </style>
</head>
<body>
    <h1>Stock Market Analysis Dashboard</h1>
    <form method="POST">
        <label>Select Stocks (hold Cmd/Ctrl for multiple):</label><br><br>
        <select name="tickers" multiple>
            {% for t in all_tickers %}
            <option value="{{ t }}" {% if t in selected %}selected{% endif %}>{{ t }}</option>
            {% endfor %}
        </select>
        <div class="buttons">
            <button name="action" value="returns">Cumulative Returns</button>
            <button name="action" value="moving_avg">Moving Averages</button>
            <button name="action" value="volatility">Volatility</button>
            <button name="action" value="correlation">Correlation Heatmap</button>
            <button name="action" value="summary">Summary Statistics</button>
        </div>
    </form>
    {% if chart %}<img src="data:image/png;base64,{{ chart }}">{% endif %}
    {% if table %}{{ table|safe }}{% endif %}
</body>
</html>
"""

def fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return encoded

def get_data(tickers):
    data = df[df['Name'].isin(tickers)].copy()
    data['daily_return'] = data.groupby('Name')['close'].pct_change()
    return data

@app.route('/', methods=['GET', 'POST'])
def index():
    chart, table = None, None
    selected = ['AAPL', 'GOOGL', 'MSFT']

    if request.method == 'POST':
        selected = request.form.getlist('tickers')
        action = request.form.get('action')

        if selected:
            data = get_data(selected)

            if action == 'returns':
                data['cumulative_return'] = data.groupby('Name')['daily_return'].transform(
                    lambda x: (1 + x).cumprod() - 1)
                fig, ax = plt.subplots(figsize=(10, 5))
                for t in selected:
                    s = data[data['Name'] == t]
                    ax.plot(s['date'], s['cumulative_return'], label=t)
                ax.set_title('Cumulative Returns')
                ax.set_ylabel('Return')
                ax.legend()
                chart = fig_to_b64(fig)

            elif action == 'moving_avg':
                data['ma20'] = data.groupby('Name')['close'].transform(lambda x: x.rolling(20).mean())
                data['ma50'] = data.groupby('Name')['close'].transform(lambda x: x.rolling(50).mean())
                fig, axes = plt.subplots(len(selected), 1, figsize=(10, 4 * len(selected)), squeeze=False)
                for i, t in enumerate(selected):
                    s = data[data['Name'] == t]
                    axes[i][0].plot(s['date'], s['close'], label='Close', alpha=0.6)
                    axes[i][0].plot(s['date'], s['ma20'], label='20-day MA')
                    axes[i][0].plot(s['date'], s['ma50'], label='50-day MA')
                    axes[i][0].set_title(f'{t} — Moving Averages')
                    axes[i][0].legend()
                fig.tight_layout()
                chart = fig_to_b64(fig)

            elif action == 'volatility':
                data['volatility'] = data.groupby('Name')['daily_return'].transform(
                    lambda x: x.rolling(30).std())
                fig, ax = plt.subplots(figsize=(10, 5))
                for t in selected:
                    s = data[data['Name'] == t]
                    ax.plot(s['date'], s['volatility'], label=t)
                ax.set_title('30-Day Rolling Volatility')
                ax.set_ylabel('Std Dev of Daily Returns')
                ax.legend()
                chart = fig_to_b64(fig)

            elif action == 'correlation':
                pivot = data.pivot_table(index='date', columns='Name', values='daily_return')
                corr = pivot.corr()
                fig, ax = plt.subplots(figsize=(6, 5))
                im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
                plt.colorbar(im, ax=ax)
                ax.set_xticks(range(len(selected)))
                ax.set_yticks(range(len(selected)))
                ax.set_xticklabels(selected)
                ax.set_yticklabels(selected)
                for i in range(len(selected)):
                    for j in range(len(selected)):
                        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')
                ax.set_title('Return Correlation Heatmap')
                chart = fig_to_b64(fig)

            elif action == 'summary':
                summary = data.groupby('Name')['daily_return'].agg(
                    Mean_Return='mean',
                    Volatility='std',
                    Total_Return=lambda x: (1 + x).prod() - 1
                ).round(4)
                table = summary.to_html()

    return render_template_string(HTML, all_tickers=all_tickers, selected=selected,
                                  chart=chart, table=table)

t = threading.Thread(target=lambda: app.run(port=5050, use_reloader=False))
t.daemon = True
t.start()

print('Dashboard running at http://localhost:5050')
display(IFrame('http://localhost:5050', width='100%', height=650))